# Vietnamese Caselaw PDF → Markdown via Marker (Surya OCR)
#
Uses the **Marker** pipeline (built on **Surya** VLM) to convert scanned
Vietnamese legal PDF documents to clean markdown.
#
Marker handles the full pipeline:
- Text detection & OCR (Surya)
- Layout analysis & reading order
- Table recognition
- Formatting & cleanup
#
Designed for **Kaggle T4 GPU** (16 GB VRAM).

In [ ]:
import subprocess, sys, os

def run(cmd):
    print(f">>> {cmd}")
    subprocess.check_call(cmd, shell=True)

run(f"{sys.executable} -m pip install -q marker-pdf")

print("\n✓ marker-pdf installed!")

>>> /usr/bin/python3 -m pip install -q marker-pdf


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import gc
import time
import glob

import torch

# ── Paths ────────────────────────────────────────────────────────────────
INPUT_DIR  = "/content"   # ← adjust to your dataset
OUTPUT_DIR = "ocr_marker"

os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU    : {gpu.name} ({gpu.total_memory / 1024**3:.1f} GB)")

pdf_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.pdf")))
print(f"\nFound {len(pdf_files)} PDF(s) in {INPUT_DIR}")

Device : cuda:0
GPU    : Tesla T4 (14.6 GB)

Found 40 PDF(s) in /content


In [5]:
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
from marker.config.parser import ConfigParser

config = {
    "output_format": "markdown",
    "force_ocr": True,
    "paginate_output": False,
    "languages": "vi",
    "batch_multiplier": 2,
    "pdf_dpi": 300,  # Try 300 or 400 (Default is usually around 150-200)
    "max_pages": None
}
config_parser = ConfigParser(config)

print("Loading Surya models …")
t0 = time.time()
artifact_dict = create_model_dict()
converter = PdfConverter(
    config=config_parser.generate_config_dict(),
    artifact_dict=artifact_dict,
    processor_list=config_parser.get_processors(),
    renderer=config_parser.get_renderer(),
)
print(f"✓ Models loaded in {time.time() - t0:.1f}s")

if torch.cuda.is_available():
    vram = torch.cuda.memory_allocated() / 1024**3
    print(f"  VRAM used: {vram:.1f} GB")

Loading Surya models …


✓ Models loaded in 59.2s
  VRAM used: 3.2 GB


In [ ]:
print("=" * 65)
print("MARKER OCR  (Surya VLM)")
print("=" * 65)

results = []
for idx, pdf_path in enumerate(pdf_files, 1):
    fname = os.path.basename(pdf_path)
    out_path = os.path.join(OUTPUT_DIR, fname.replace(".pdf", ".md"))

    # Skip if already processed
    if os.path.exists(out_path):
        print(f"[{idx}/{len(pdf_files)}] SKIP (exists) {fname}")
        results.append({"file": fname, "status": "SKIP", "time_s": 0})
        continue

    print(f"[{idx}/{len(pdf_files)}] {fname}")
    t0 = time.time()
    try:
        rendered = converter(pdf_path)
        text, metadata, images = text_from_rendered(rendered)
        dt = time.time() - t0

        if text and text.strip():
            with open(out_path, "w", encoding="utf-8") as f:
                f.write(text)
            status = "OK"
            print(f"  → OK  {len(text)} chars ({dt:.1f}s)")
        else:
            status = "EMPTY"
            print(f"  → EMPTY ({dt:.1f}s)")

    except Exception as e:
        dt = time.time() - t0
        status = "FAIL"
        print(f"  → FAIL: {e} ({dt:.1f}s)")

    results.append({"file": fname, "status": status, "time_s": round(dt, 1)})
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

ok = sum(1 for r in results if r["status"] == "OK")
print(f"\nMarker done: {ok}/{len(results)} OK\n")

MARKER OCR  (Surya VLM)
[1/40] 04-06-2025-Cao_Bang-2ta1957263t1cvn.pdf


Recognizing Text: 100%|██████████| 151/151 [01:42<00:00,  1.47it/s]
Detecting bboxes: 0it [00:00, ?it/s]


  → OK  17036 chars (118.8s)
[2/40] 05-06-2025-Cao_Bang-2ta1956999t1cvn.pdf


Recognizing Text: 100%|██████████| 132/132 [01:48<00:00,  1.22it/s]
Detecting bboxes: 0it [00:00, ?it/s]


  → OK  21253 chars (126.1s)
[3/40] 06-06-2025-Cao_Bang-2ta1957550t1cvn.pdf


...

Recognizing Text: 100%|██████████| 98/98 [02:10<00:00,  1.33s/it]
Detecting bboxes: 0it [00:00, ?it/s]


  → OK  16511 chars (145.4s)
[16/40] 18-08-2025-Cao_Bang-2ta1958073t1cvn.pdf


Recognizing Text: 100%|██████████| 80/80 [02:35<00:00,  1.95s/it]
Detecting bboxes: 0it [00:00, ?it/s]


  → OK  11371 chars (167.7s)
[17/40] 18-08-2025-Ho_Chi_Minh-2ta1957575t1cvn.pdf


Recognizing Text:   3%|▎         | 3/86 [00:25<08:54,  6.44s/it]

In [7]:
del converter, artifact_dict
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("GPU memory freed.")

GPU memory freed.


In [8]:
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

ok    = sum(1 for r in results if r["status"] == "OK")
empty = sum(1 for r in results if r["status"] == "EMPTY")
fail  = sum(1 for r in results if r["status"] == "FAIL")
skip  = sum(1 for r in results if r["status"] == "SKIP")
total = len(results)
total_time = sum(r["time_s"] for r in results)

print(f"\n{'OK':>5} {'Empty':>6} {'Fail':>6} {'Skip':>6} {'Total':>6}")
print("-" * 35)
print(f"{ok:>5} {empty:>6} {fail:>6} {skip:>6} {total:>6}")
print(f"\nTotal time: {total_time/60:.1f} min")
print(f"Outputs saved to: {OUTPUT_DIR}/")

FINAL SUMMARY

   OK  Empty   Fail   Skip  Total
-----------------------------------
   20      0      0      0     20

Total time: 57.3 min
Outputs saved to: ocr_marker/


In [9]:
md_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.md")))
if md_files:
    with open(md_files[0], "r", encoding="utf-8") as f:
        content = f.read()
    print(f"Preview: {os.path.basename(md_files[0])}")
    print("-" * 50)
    print(content[:3000])
    if len(content) > 3000:
        print(f"\n… ({len(content) - 3000} more characters)")

Preview: 09-06-2025-Gia_Lai-2ta1849797t1cvn.md
--------------------------------------------------
## TÒA ÁN NHÂN DÂN HUYỆN CHƯ PRÔNG TỈNH GIA LAI

# CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc

Bản án số: 26/2025/HS-ST

Ngày 09-6-2025

## NHÂN DANH NƯỚC CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM

## TÒA ÁN NHÂN DÂN HUYỆN CHƯ PRÔNG, TỈNH GIA LAI

- Thành phần Hội đồng xét xử sơ thẩm gồm có:

Thẩm phán - Chủ tọa phiên tòa: Ông Trương Nam Trung.

Các Hội thẩm nhân dân: Ông Trương Công Chự và ông Nguyễn Ngọc Bình

- Thư ký phiên tòa: Ông **Trần Long Tuân** Thư ký Tòa án nhân dân huyện Chư Prông, tỉnh Gia Lai.
- Đại diện Viện kiểm sát nhân dân huyện Chư Prông, tỉnh Gia Lai tham gia phiên tòa: Bà Mai Thị Bích Ngọc- Kiểm sát viên.

Ngày 09 tháng 6 năm 2024 tại trụ sở Tòa án nhân dân huyện Chư Prông, tỉnh Gia Lai xét xử sơ thẩm công khai vụ án hình sự sơ thẩm thụ lý số: 20/2025/TLST-HS ngày 13 tháng 5 năm 2025 theo Quyết định đưa vụ án ra xét xử số: 18/2025/QĐXXST-HS ngày 16 tháng 5